## **11_lm_head: The Grand Finale: The Language Model Head**

We have reached the final architectural component of our GPT model.  
Our data has traveled a long journey: embeddings → deep stack of transformer blocks → final layer norm.  
We now have highly context-aware vectors, one for each token. Shape: `(B, T, C)`.

But this is just an internal representation — **not a prediction**.  
How do we use these rich vectors to predict the single next word?

### The `lm_head`: A Parallel Prediction Layer

```python
self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
self.lm_head.weight = self.wte.weight
```

It's just a simple `nn.Linear` layer that projects from our internal vector space (`C` dimensions) into the **vocabulary space** (`vocab_size` dimensions).

The brilliant twist: this projection is applied **independently and in parallel** to every token's vector along the `T` dimension.

| Variable | Input Shape | Output Shape | Meaning |
| :--- | :--- | :--- | :--- |
| `logits` | `(B, T, C)` | `(B, T, vocab_size)` | A raw score for every possible next token, at **each** position |

In [ ]:
import torch
import torch.nn as nn

B, T, C = 1, 4, 64
vocab_size = 100

# Pretend this is our final processed tensor from ln_f
x = torch.randn(B, T, C)

# The language model head
lm_head = nn.Linear(C, vocab_size, bias=False)

logits = lm_head(x)
print("Input shape:  ", x.shape)
print("Output shape: ", logits.shape)
print(f"\nFor each of {T} tokens, we get {vocab_size} scores (one per vocab word)")

### "Why make `T` predictions? Isn't that wasteful?"

This is one of the most brilliant design choices in the Transformer.  
The purpose depends on whether you are **training** or **generating**.

#### 1. For Efficient Training

During training, we want the model to predict the next word at **every position**, all at once.

Input: "A crane ate fish" (`T=4`)

+ `logits[:, 0, :]` → prediction based on "A" → target: "crane"
+ `logits[:, 1, :]` → prediction based on "A crane" → target: "ate"
+ `logits[:, 2, :]` → prediction based on "A crane ate" → target: "fish"

Thanks to the causal mask, each position only uses past context.  
All predictions in **one forward pass**. Incredibly efficient for GPU training.

#### 2. For Generation (Inference)

When generating, we are seemingly "wasteful":

+ Input: "A crane ate" (`T=3`)
+ Logits shape: `(1, 3, vocab_size)`
+ We **throw away** positions 0 and 1
+ We **only use** `logits[:, -1, :]` to sample the next word

This is a pragmatic trade-off: the architecture is optimized for the massively parallel training.  
We leverage that same architecture for inference.

In [ ]:
# During generation: we only care about the LAST position
last_logits = logits[:, -1, :]  # (B, vocab_size)
print("Logits at last position shape:", last_logits.shape)

# Convert to probabilities
probs = torch.softmax(last_logits, dim=-1)
print("Probabilities shape:", probs.shape)
print(f"Sum of probabilities: {probs.sum().item():.4f}")

# Sample the next token
next_token = torch.multinomial(probs, num_samples=1)
print(f"\nPredicted next token ID: {next_token.item()}")

### The Final Trick: Weight Tying

```python
self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
self.lm_head.weight = self.wte.weight  # ← THIS LINE
```

The second line is a simple but profound optimization called **weight tying**.  
It tells PyTorch that the `lm_head` weight is the **exact same object in memory** as the `wte` weight.  
When backpropagation updates one, it simultaneously updates the other.

### Why does weight tying make sense?

| Layer | Shape `(rows, cols)` | Role |
| :--- | :--- | :--- |
| `wte.weight` | `(vocab_size, n_embd)` | **ID → Meaning**: Convert token ID to meaning vector |
| `lm_head.weight` | `(vocab_size, n_embd)` | **Meaning → ID**: Convert meaning vector to scores |

They have the **exact same shape** and their functions are perfectly **symmetric**:  
one goes from ID to meaning, the other from meaning back to ID.

The core insight: the vector that represents the meaning of a word should be the **same**  
whether that word is an input or an output.

In [ ]:
# Demonstrate weight tying
wte = nn.Embedding(vocab_size, C)
lm_head = nn.Linear(C, vocab_size, bias=False)

# TIE the weights
lm_head.weight = wte.weight

# They are the SAME object in memory
print("Are weights the same object?", lm_head.weight is wte.weight)
print("wte shape:    ", wte.weight.shape)
print("lm_head shape:", lm_head.weight.shape)

# Count the savings
params_without_tying = vocab_size * C * 2  # two separate matrices
params_with_tying = vocab_size * C          # one shared matrix
print(f"\nWithout tying: {params_without_tying:,} parameters")
print(f"With tying:    {params_with_tying:,} parameters")
print(f"Saved:         {params_without_tying - params_with_tying:,} parameters")

### Benefits of Weight Tying

1. **Massive Parameter Reduction**: For GPT-2 small, this saves ~38.5 million parameters (`50,257 × 768`) with one line of code
2. **Improved Performance**: Acts as a powerful regularization — enforcing a sensible architectural constraint prevents overfitting

### The Complete GPT-2 Model

In [ ]:
import torch.nn.functional as F
import math
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 100
    block_size: int = 32
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.drop(self.proj(x))
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Part 1: Input Layers
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)

        # Part 2: Core Processing
        self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])

        # Part 3: Output
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # WEIGHT TYING
        self.lm_head.weight = self.wte.weight

    def forward(self, idx):
        B, T = idx.size()
        tok_emb = self.wte(idx)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.wpe(pos)
        x = self.drop(tok_emb + pos_emb)

        for block in self.h:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

# Build and test the COMPLETE model
config = GPTConfig()
model = GPT2(config)

idx = torch.randint(0, config.vocab_size, (2, 16))
logits = model(idx)

print("Input (token IDs) shape:", idx.shape)
print("Output (logits) shape: ", logits.shape)
print(f"\nFor each of {idx.shape[1]} tokens, we get {config.vocab_size} scores")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print("Weight tying confirmed:", model.lm_head.weight is model.wte.weight)

### We built it. The complete GPT-2 architecture.

Every single piece — from token embeddings to the language model head — built from the ground up.

```
Token IDs → wte + wpe → dropout
    → Block 0 (LN → Attention → + → LN → MLP → +)
    → Block 1 ...
    → Block N ...
    → ln_f → lm_head → Logits (vocab_size scores per position)
```

These roughly 100 lines of Python contain **all** the core architectural ideas behind multi-billion dollar models like GPT.  
There is no hidden code, no secret sauce.